# Prepare data for EGA submission 

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
            
            # Seurat
            library(Seurat), 

            # Data 
            library(tidyverse), 
            library(data.table), 
            
            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [3]:
# Configure reticulate 
use_condaenv(condaenv='p.3.9.19-FD20200109SPLENO', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

python:         /nobackup/peer/fdeckert/miniconda3/envs/p.3.9.19-FD20200109SPLENO/bin/python
libpython:      /nobackup/peer/fdeckert/miniconda3/envs/p.3.9.19-FD20200109SPLENO/lib/libpython3.9.so
pythonhome:     /nobackup/peer/fdeckert/miniconda3/envs/p.3.9.19-FD20200109SPLENO:/nobackup/peer/fdeckert/miniconda3/envs/p.3.9.19-FD20200109SPLENO
version:        3.9.19 | packaged by conda-forge | (main, Mar 20 2024, 12:50:21)  [GCC 12.3.0]
numpy:          /nobackup/peer/fdeckert/miniconda3/envs/p.3.9.19-FD20200109SPLENO/lib/python3.9/site-packages/numpy
numpy_version:  1.26.4

NOTE: Python version was forced by use_python() function

In [4]:
random_seed <- 42
set.seed(random_seed)

In [5]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20220623HSCT")

# Parameter 

In [6]:
meta_select <- c("nCount_RNA", "nFeature_RNA", "patient_id", "time_point", "hto_demux_class", "digestion_protocol", "pMT_RNA", "pHB_RNA", "pRP_RNA", "msS_RNA", "msG2M_RNA", "msCC_diff_RNA", "cc_phase_class", "celltypist_predicted_labels_low", "celltypist_predicted_labels_high", "genotype_class", "has_ir", "cc_aa_alignment", "cc_aa_alignment_size", "cell_type_leiden_subset")

# Seurat 

In [7]:
so <- readRDS("data/object/pp_subset.rds")

In [8]:
so@meta.data <- so@meta.data[, meta_select] %>% 
    dplyr::rename(cell_type_leiden=cell_type_leiden_subset) %>% 
    dplyr::mutate(patient_id=as.character(patient_id)) %>% 
    dplyr::mutate(patient_id=ifelse(patient_id=="Patient_10", "Patient_9", patient_id)) %>% 
    mutate(patient_id=ifelse(patient_id=="Patient_11", "Patient_10", patient_id))

In [9]:
saveRDS(so, "/nobackup/peer/fdeckert/FD20220623HSCT/data/ZENODO/seurat.rds")

# AnnData

In [10]:
# Store data as h5ad 
ad <- import("anndata", as="ad", convert=FALSE)
pd <- import("pandas", as="pd", convert=FALSE)
np <- import("numpy", as="np", convert=FALSE)
    
# Transform dgCMatrix to sparse sc_sparse matrix
adata <- ad$read_h5ad("data/object/pp_subset.h5ad")
adata$obs <- adata$obs$filter(meta_select)$rename(columns =list("cell_type_leiden_subset"="cell_type_leiden"))
adata$obs[['patient_id']] <- adata$obs[['patient_id']]$replace("Patient_10", "Patient_9")
adata$obs[['patient_id']] <- adata$obs[['patient_id']]$replace("Patient_11", "Patient_10")

adata$write_h5ad("/nobackup/peer/fdeckert/FD20220623HSCT/data/ZENODO/anndata.h5ad")

None